# 02 — Naive on/off plus-minus (the baseline, and why it fails)

The intuitive metric this project started from: *how does the team's goal
difference per 90 change when a player is on the pitch vs off it?*

$$\text{naive}_p = \underbrace{\frac{GD_{on}}{min_{on}} \cdot 90}_{\text{on rate}} - \underbrace{\frac{GD_{off}}{min_{off}} \cdot 90}_{\text{off rate}}$$

It is computed honestly here — then we look at what it gets wrong.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from plimpact.config import load_config
from plimpact.validate import load_tables

cfg = load_config()
tables = load_tables(cfg)
matches, appearances, shots, stints = (
    tables["matches"], tables["appearances"], tables["shots"], tables["stints"]
)
pd.set_option("display.width", 120)

In [2]:
from plimpact.naive import naive_plus_minus

naive = naive_plus_minus(cfg, matches, stints)
players = pd.read_parquet(cfg.processed_dir / "players.parquet")
naive = naive.merge(players[["player_id", "player"]], on="player_id")
qualified = naive[naive["on_minutes"] >= cfg.min_minutes]
cols = ["player", "team", "on_minutes", "on_gd90", "off_gd90", "naive_gd90"]
qualified.sort_values("naive_gd90", ascending=False).head(15)[cols].round(2)

,player,team,on_minutes,on_gd90,off_gd90,naive_gd90
335,Mateus Fernandes,Southampton,2978,-1.27,-3.45,2.18
582,Dominic Calvert-Lewin,Leeds,2748,0.16,-1.54,1.70
882,Bruno Guimarães,Newcastle United,9133,0.53,-0.89,1.42
214,Gustavo Hamer,Sheffield United,3001,-1.62,-3.03,1.41
536,Emiliano Martinez,Aston Villa,9134,0.41,-0.97,1.38
981,Elliot Anderson,Newcastle United,1006,1.61,0.23,1.38
412,Chemsdine Talbi,Sunderland,1567,0.57,-0.76,1.34
266,Bruno Fernandes,Manchester United,9298,0.20,-1.13,1.33
1074,Michael Olise,Crystal Palace,1291,1.05,-0.26,1.30
811,George Baldock,Sheffield United,972,-0.93,-2.15,1.22


## Failure mode 1 — small-sample extremes

Drop the minutes filter and the "best players in the league" become squad
players who happened to be on for a few good stints:

In [3]:
low = naive[(naive["on_minutes"] >= 180) & (naive["on_minutes"] < 900)]
low.sort_values("naive_gd90", ascending=False).head(8)[cols].round(2)

,player,team,on_minutes,on_gd90,off_gd90,naive_gd90
679,Reiss Nelson,Arsenal,222,3.65,1.18,2.47
187,Chiedozie Ogbene,Ipswich,237,0.76,-1.35,2.11
56,Manor Solomon,Tottenham,196,1.84,-0.01,1.85
294,Tyrique George,Chelsea,307,2.05,0.31,1.75
103,Fábio Vieira,Arsenal,280,2.89,1.18,1.71
762,Emile Smith-Rowe,Arsenal,329,2.74,1.18,1.56
1002,Nathan Broadhead,Ipswich,687,0.00,-1.50,1.50
430,Jair,Nottingham Forest,541,1.33,-0.16,1.49


## Failure mode 2 — teammates are inseparable

Players who share most of their pitch time get nearly the same rating; on/off
cannot tell them apart. For each qualified player, find the teammate they share
the highest fraction of minutes with:

In [4]:
from plimpact.naive import stint_team_view

tv = stint_team_view(matches, stints)
long = tv.explode("players").rename(columns={"players": "player_id"})
# pairwise shared minutes within one example team
team = qualified.sort_values("naive_gd90", ascending=False).iloc[0]["team"]
tl = long[long["team"] == team]
pivot = tl.pivot_table(index="player_id", columns=["match_id", "stint_idx"],
                       values="duration", aggfunc="sum").fillna(0)
mins = pivot.sum(axis=1)
top_ids = mins.nlargest(8).index
shared = (pivot.loc[top_ids] > 0).astype(int) @ pivot.loc[top_ids].T.clip(lower=0)
name_of = players.set_index("player_id")["player"]
overlap = pd.DataFrame(shared.values / mins[top_ids].values[:, None],
                       index=name_of[top_ids], columns=name_of[top_ids])
overlap.round(2)

player,Mateus Fernandes,Kyle Walker-Peters,Taylor Harwood-Bellis,Aaron Ramsdale,Jan Bednarek,Flynn Downes,Joe Ayodele-Aribo,Tyler Dibling
player,,,,,,,,
Mateus Fernandes,1.00,0.84,0.80,0.81,0.72,0.63,0.55,0.56
Kyle Walker-Peters,0.85,1.00,0.83,0.76,0.77,0.58,0.63,0.52
Taylor Harwood-Bellis,0.84,0.86,1.00,0.78,0.80,0.67,0.58,0.58
Aaron Ramsdale,0.89,0.82,0.82,1.00,0.81,0.62,0.55,0.55
Jan Bednarek,0.84,0.89,0.89,0.86,1.00,0.65,0.59,0.50
Flynn Downes,0.85,0.78,0.87,0.77,0.76,1.00,0.47,0.52
Joe Ayodele-Aribo,0.80,0.91,0.81,0.73,0.74,0.51,1.00,0.57
Tyler Dibling,0.88,0.82,0.88,0.79,0.68,0.61,0.62,1.00


Most starters at one club share 70%+ of their minutes — their naive ratings are
statements about the *team*, not the player.

## Failure mode 3 — substitution context

Subs disproportionately enter when the game state is already decided (protecting
or chasing), so their on-minutes are not a random sample of match situations.

**All three problems have the same fix:** regress stint outcomes on *everyone on
the pitch simultaneously*, so each player is credited only with what their
presence adds — next notebook.